# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 01.01 · Adquisición incremental de subtítulos

Reutiliza transcripciones canónicas y cachés por `video_id`; solo consulta YouTube para candidatos nuevos y nunca descarga audio o video.

La adquisición nueva usa `yt-dlp` para localizar pistas de subtítulos [1]. Las transcripciones automáticas se conservan como insumo imperfecto, no como verdad textual, porque se han documentado sesgos de dialecto y género en el subtitulado automático de YouTube [2]. Toda ampliación debe respetar los términos de la plataforma [3] y la evaluación ética contextual recomendada para investigación en Internet [4]. La reutilización de cachés y la selección de candidatos son decisiones locales registradas en manifiestos. El modo dirigido prioriza cobertura insuficiente siguiendo criterios de aprendizaje activo y desbalance multietiqueta [5] [6], pero no permite estimar prevalencias en YouTube ni en el Perú.

**Contrato v2.1:** `SEGURO` + cuatro daños entrenados, incluida `ATAQUE_POR_GENERO_IDENTIDAD`. `SEGURO` es excluyente; los daños son multietiqueta. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [ ]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('Proyecto:', ROOT)


## Preflight

In [ ]:
from moderacion_peru.artifacts import artifact_status
artifact_status(ROOT)

## Parámetros editables

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONTROLES DEL SCRAPING: edite únicamente este bloque
# ══════════════════════════════════════════════════════════════════════════════
DISCOVER_NEW = False          # True: consulta canales/búsquedas y guarda candidatos
FETCH_NEW = False             # True: obtiene subtítulos solo de videos aún no procesados
DISCOVERY_MODE = "seed"       # "seed", "directed" o "both"

MAX_NEW_VIDEOS = 50           # máximo de llamadas nuevas para obtener subtítulos
MAX_VIDEOS_PER_CHANNEL = 75   # candidatos recientes inspeccionados por canal
MAX_RESULTS_PER_QUERY = 20    # candidatos inspeccionados por consulta dirigida

SUBTITLE_LANGUAGES = ("es-PE", "es-419", "es")
YT_RETRIES = 3
YT_SLEEP_MIN_SECONDS = 1.0
YT_SLEEP_MAX_SECONDS = 3.0
STOP_ON_VIDEO_ERROR = False   # False: registra el fallo y continúa con el siguiente

if DISCOVERY_MODE not in {"seed", "directed", "both"}:
    raise ValueError("DISCOVERY_MODE debe ser seed, directed o both")
if MAX_NEW_VIDEOS < 0 or MAX_VIDEOS_PER_CHANNEL < 1 or MAX_RESULTS_PER_QUERY < 1:
    raise ValueError("Los límites de videos deben ser válidos")
if YT_SLEEP_MIN_SECONDS < 0 or YT_SLEEP_MAX_SECONDS < YT_SLEEP_MIN_SECONDS:
    raise ValueError("El intervalo de espera de yt-dlp no es válido")

print({
    "discover_new": DISCOVER_NEW,
    "fetch_new": FETCH_NEW,
    "discovery_mode": DISCOVERY_MODE,
    "max_new_videos": MAX_NEW_VIDEOS,
    "max_videos_per_channel": MAX_VIDEOS_PER_CHANNEL,
    "max_results_per_query": MAX_RESULTS_PER_QUERY,
    "subtitle_languages": SUBTITLE_LANGUAGES,
})


## Canales y consultas

In [ ]:
# Canales semilla recuperados del cuaderno histórico; puede añadir, quitar o editar filas.
# `categoria_fuente` describe el dominio/registro del canal, no una etiqueta de daño.
_SEED_ROWS = [
    ("Marco Sifuentes / Ocram", "politica_analisis", "https://www.youtube.com/@canalYAAAAA"),
    ("El diario de Curwen", "politica_opinion", "https://www.youtube.com/@curwen"),
    ("Sin Guion con Rosa María Palacios", "politica_periodismo", "https://www.youtube.com/@singuionlr"),
    ("RPP Noticias", "politica_actualidad", "https://www.youtube.com/@RPPNoticias"),
    ("Exitosa Noticias", "politica_actualidad", "https://www.youtube.com/@exitosape"),
    ("Willax Television", "politica_opinion", "https://www.youtube.com/@WillaxTV"),
    ("Canal N", "politica_actualidad", "https://www.youtube.com/@canaln"),
    ("ATV Noticias", "politica_actualidad", "https://www.youtube.com/@ATVNoticias"),
    ("Latina Noticias", "politica_actualidad", "https://www.youtube.com/@latinanoticias"),
    ("Panamericana Noticias", "politica_actualidad", "https://www.youtube.com/@Panamericana-Noticias"),
    ("Hablando Huevadas", "humor_streaming", "https://www.youtube.com/@HablandoHuevadasOficial"),
    ("Todo Good", "streaming_opinion", "https://www.youtube.com/@todogoodpe"),
    ("Goblinciano", "streaming_opinion", "https://www.youtube.com/@Goblinciano"),
    ("El Cacas", "humor_streaming", "https://www.youtube.com/@ElCacas"),
    ("Negro Fuertes", "humor_comedia", "https://www.youtube.com/@NegroFuertes"),
    ("Jason Qqq", "humor_streaming", "https://www.youtube.com/@JasonQqqOficial"),
    ("La Cotorrisa Perú", "humor_podcast", "https://www.youtube.com/@LaCotorrisaPeru"),
    ("Magaly TV La Firme", "farandula", "https://www.youtube.com/@MagalyTVLaFirmeATV"),
    ("Amor y Fuego", "farandula", "https://www.youtube.com/@AmoryFuego"),
    ("América Hoy", "farandula", "https://www.youtube.com/@americahoytv"),
    ("Instarándula", "farandula_digital", "https://www.youtube.com/@Instarandula"),
    ("El Popular", "farandula_digital", "https://www.youtube.com/@ElPopularPeru"),
    ("Nico Moschella", "deportes_informal", "https://www.youtube.com/@NicoMoschella"),
    ("Líbero Deportes", "deportes", "https://www.youtube.com/@DiarioLiberoOficial"),
    ("Depor", "deportes", "https://www.youtube.com/@DeporPeru"),
    ("Misias pero viajeras", "viajes", "https://www.youtube.com/c/Misiasperoviajeras"),
    ("Buen Viaje", "viajes", "https://www.youtube.com/c/BuenViajePe"),
    ("Viaja y Prueba", "viajes_gastronomia", "https://www.youtube.com/@ViajayPrueba"),
    ("Cocinando con la Patty", "gastronomia", "https://www.youtube.com/@CocinandoConLaPatty"),
    ("Arde Troya con Juliana Oxenford", "politica_analisis", "https://www.youtube.com/@ardetroyalr"),
    ("Panorama", "politica_periodismo", "https://www.youtube.com/@PanoramaPTV"),
    ("Juanito y Richard", "humor_comedia", "https://www.youtube.com/@JuanitoyRichard"),
    ("Nada Espacial", "humor_podcast", "https://www.youtube.com/@nadaespacialpodcast"),
    ("L1MAX", "deportes_informal", "https://www.youtube.com/@L1MAX_"),
    ("Cocina Cajamarquina", "gastronomia", "https://www.youtube.com/@cocinacajamarquina"),
    ("Tío Lenguado y Descocaos", "viajes_gastronomia", "https://www.youtube.com/@tiolenguado"),
]
SEED_CHANNELS = [
    {"name": name, "categoria_fuente": category, "url": url}
    for name, category, url in _SEED_ROWS
]

# Ampliación dirigida: las cuotas son máximos por canal, nunca prevalencias esperadas.
DIRECTED_CHANNELS = [
    {"name": "Hablando Huevadas", "url": "https://www.youtube.com/@HablandoHuevadasOficial", "quota": 70, "target_category": "CONTENIDO_SEXUAL|ATAQUE_POR_GENERO_IDENTIDAD|ACOSO_AMENAZA"},
    {"name": "Goblinciano", "url": "https://www.youtube.com/@Goblinciano", "quota": 85, "target_category": "RACISMO_DISCRIMINACION|ACOSO_AMENAZA"},
    {"name": "Juanito y Richard", "url": "https://www.youtube.com/@JuanitoyRichard", "quota": 85, "target_category": "RACISMO_DISCRIMINACION|ACOSO_AMENAZA"},
    {"name": "Arde Troya con Juliana Oxenford", "url": "https://www.youtube.com/@ardetroyalr", "quota": 55, "target_category": "ACOSO_AMENAZA"},
    {"name": "Todo Good", "url": "https://www.youtube.com/@todogoodpe", "quota": 40, "target_category": "ACOSO_AMENAZA"},
    {"name": "Magaly TV La Firme", "url": "https://www.youtube.com/@MagalyTVLaFirmeATV", "quota": 35, "target_category": "ACOSO_AMENAZA|CONTENIDO_SEXUAL"},
]

SEED_SEARCH_QUERIES = [
    "noticias política Perú canal YouTube",
    "periodismo opinión Perú YouTube",
    "humor streaming Perú lenguaje coloquial",
    "farándula espectáculos Perú",
    "deportes peruanos comentarios YouTube",
]
DIRECTED_SEARCH_QUERIES = [
    {"query": "insultos racistas discriminación Perú denuncia", "target_category": "RACISMO_DISCRIMINACION"},
    {"query": "ataque machista misoginia Perú denuncia", "target_category": "ATAQUE_POR_GENERO_IDENTIDAD"},
    {"query": "ataque homofóbico transfóbico Perú denuncia", "target_category": "ATAQUE_POR_GENERO_IDENTIDAD"},
    {"query": "amenaza de muerte denuncia Perú", "target_category": "ACOSO_AMENAZA"},
    {"query": "extorsionadores amenazan audio Perú", "target_category": "ACOSO_AMENAZA"},
    {"query": "acoso sexual denuncia televisión peruana", "target_category": "CONTENIDO_SEXUAL|ACOSO_AMENAZA"},
]

CHANNEL_SOURCES = (
    SEED_CHANNELS if DISCOVERY_MODE == "seed"
    else DIRECTED_CHANNELS if DISCOVERY_MODE == "directed"
    else SEED_CHANNELS + DIRECTED_CHANNELS
)
SEARCH_QUERIES = (
    SEED_SEARCH_QUERIES if DISCOVERY_MODE == "seed"
    else DIRECTED_SEARCH_QUERIES if DISCOVERY_MODE == "directed"
    else SEED_SEARCH_QUERIES + DIRECTED_SEARCH_QUERIES
)
print("Canales configurados:", len(CHANNEL_SOURCES), "· consultas:", len(SEARCH_QUERIES))


## Reutilización de snapshots existentes

In [ ]:
from moderacion_peru.acquisition import (bootstrap_canonical_from_existing, discover_existing_transcript_sources, load_candidates, merge_candidates)
CANONICAL = ROOT/'datos/raw/transcripts_raw.jsonl'
CACHE = ROOT/'datos/raw/transcripts_cache'
sources = discover_existing_transcript_sources(ROOT, canonical_path=CANONICAL)
reuse_stats = bootstrap_canonical_from_existing(sources, CANONICAL)
print(reuse_stats)

## Descubrimiento general o ampliación dirigida

In [ ]:
from moderacion_peru.acquisition import discover_youtube_candidates
from moderacion_peru.io import append_jsonl_once, write_json_atomic

DISCOVERED_PATH = ROOT/'datos/raw/video_candidates.jsonl'
DISCOVERY_FAILURES_PATH = ROOT/'datos/raw/fallos_descubrimiento_ultima_ejecucion.json'
discovered = []
if DISCOVER_NEW:
    discovered, discovery_failures = discover_youtube_candidates(
        CHANNEL_SOURCES,
        SEARCH_QUERIES,
        max_videos_per_channel=MAX_VIDEOS_PER_CHANNEL,
        max_results_per_query=MAX_RESULTS_PER_QUERY,
        retries=YT_RETRIES,
        sleep_min_seconds=YT_SLEEP_MIN_SECONDS,
        sleep_max_seconds=YT_SLEEP_MAX_SECONDS,
    )
    added, existing = append_jsonl_once(DISCOVERED_PATH, discovered, id_field='video_id')
    write_json_atomic(DISCOVERY_FAILURES_PATH, discovery_failures)
    print({"discovered": len(discovered), "added": added, "existing": existing,
           "source_failures": len(discovery_failures)})
else:
    print("Descubrimiento desactivado: se reutilizan candidatos y transcripciones locales.")


## Candidatos y caché

In [ ]:
CANDIDATE_FILES = [ROOT/'datos/raw/video_candidates.jsonl', ROOT/'datos/raw/videos_candidatos.csv']
loaded_groups = [load_candidates(source) for source in CANDIDATE_FILES]
candidates = merge_candidates(*loaded_groups)
print('Candidatos únicos:', len(candidates), '· los ya existentes se omitirán')

## Ejecución controlada y tolerante a fallos

In [ ]:
from functools import partial
from moderacion_peru.acquisition import fetch_youtube_subtitles, ingest_incremental

FAILURES = ROOT/'datos/raw/fallos_adquisicion.jsonl'
fetcher = partial(
    fetch_youtube_subtitles,
    languages=SUBTITLE_LANGUAGES,
    retries=YT_RETRIES,
    sleep_min_seconds=YT_SLEEP_MIN_SECONDS,
    sleep_max_seconds=YT_SLEEP_MAX_SECONDS,
)
if candidates:
    stats = ingest_incremental(
        candidates,
        CANONICAL,
        CACHE,
        fetcher=fetcher if FETCH_NEW else None,
        failure_path=FAILURES,
        max_new_videos=MAX_NEW_VIDEOS,
        stop_on_error=STOP_ON_VIDEO_ERROR,
    )
    print(stats)
    if stats['failed']:
        print(f"Se omitieron {stats['failed']} videos inaccesibles; revise {FAILURES}.")
else:
    print("No hay candidatos. Active DISCOVER_NEW o añada un CSV/JSONL; no se descarga de nuevo el corpus.")


## Referencias

[1] yt-dlp contributors, "yt-dlp: A Feature-Rich Command-Line Audio/Video Downloader," GitHub repository, 2026. [Online]. Available: https://github.com/yt-dlp/yt-dlp. Accessed: Aug. 5, 2026.

[2] R. Tatman, "Gender and Dialect Bias in YouTube's Automatic Captions," in Proc. 1st ACL Workshop Ethics NLP, 2017, pp. 53–59, doi: 10.18653/v1/W17-1606.

[3] YouTube, "Terms of Service," Nov. 2023. [Online]. Available: https://www.youtube.com/t/terms. Accessed: Aug. 5, 2026.

[4] A. S. franzke, A. Bechmann, M. Zimmer, et al., "Internet Research: Ethical Guidelines 3.0," Association of Internet Researchers, 2020. [Online]. Available: https://aoir.org/reports/ethics3.pdf

[5] Y. Fairstein, O. Kalinsky, Z. Karnin, et al., "Class Balancing for Efficient Active Learning in Imbalanced Datasets," in Proc. 18th Linguistic Annotation Workshop, 2024, pp. 77–86, doi: 10.18653/v1/2024.law-1.8.

[6] Y. Huang, B. Giledereli, A. Köksal, et al., "Balancing Methods for Multi-label Text Classification with Long-Tailed Class Distribution," in Proc. EMNLP, 2021, pp. 8153–8161, doi: 10.18653/v1/2021.emnlp-main.643.